In [ ]:
!git clone https://github.com/venkatsaikondra/CNN_ViT_Hybrid_Vit_Research_Pneumonia_4_Classification.git

In [ ]:
import torch
import torch.nn as nn
import timm
from torchvision import models

class ResearchHybridModel(nn.Module):
    def __init__(self, num_classes=4, dropout_rate=0.4):
        super(ResearchHybridModel, self).__init__()

        # 1. Feature Extractor (Backbone)
        # Using ResNet18 for higher efficiency (Better for 'Lightweight' claims)
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])

        # 2. Transformer Core
        # Tiny ViT ensures the model can be deployed on edge devices
        self.vit = timm.create_model('vit_tiny_patch16_224', pretrained=True)

        # 3. The 'Bridge': Projecting 512 CNN channels to 192 ViT tokens
        self.projector = nn.Conv2d(512, 192, kernel_size=1)

        # 4. Uncertainty-Aware Classifier
        self.dropout = nn.Dropout(p=dropout_rate)
        self.classifier = nn.Linear(192, num_classes)

    def forward(self, x):
        # Extract Local Features
        local_feat = self.backbone(x)

        # Bridge to Transformer
        x = self.projector(local_feat)
        x = x.flatten(2).transpose(1, 2)

        # Global Context Attention
        x = self.vit.blocks(x)
        x = self.vit.norm(x)

        # Global Representation
        global_feat = x.mean(dim=1)

        # Output with Dropout (Active during training and inference for UQ)
        return self.classifier(self.dropout(global_feat))

model = ResearchHybridModel(num_classes=4).to('cuda')

In [ ]:
def train_model(model, train_loader, val_loader, epochs=25):
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(epochs):
        model.train()
        # standard training loop logic...
        # (See previous training code provided)
        scheduler.step()

# UQ Inference Function
def get_prediction_with_uncertainty(model, img_tensor, iterations=10):
    model.train() # Keep dropout active during inference
    preds = []
    for _ in range(iterations):
        with torch.no_grad():
            output = model(img_tensor)
            preds.append(torch.softmax(output, dim=1))

    stacked_preds = torch.stack(preds)
    mean_prediction = stacked_preds.mean(dim=0)
    uncertainty_score = stacked_preds.std(dim=0).mean().item()

    return mean_prediction, uncertainty_score

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

def final_test_evaluation(model, test_loader):
    model.eval()
    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs.to('cuda'))
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    # 1. Classification Table
    print(classification_report(all_labels, all_preds, target_names=['Covid', 'Normal', 'Bact', 'Viral']))

    # 2. Multi-class AUC (Vital for Medical Papers)
    auc = roc_auc_score(all_labels, all_probs, multi_class='ovr')
    print(f"Total AUROC: {auc:.4f}")

    # 3. Confusion Matrix (For Figure 5)
    cm = confusion_matrix(all_labels, all_preds)
    return cm